In [ ]:
using_colab = True

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'

    #!mkdir -p images
    #!wget -P images https://raw.githubusercontent.com/facebookresearch/sam2/main/notebooks/images/cars.jpg

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda"
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

In [ ]:
np.random.seed(3)

def show_anns(anns, borders=True):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)

    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.5]])
        img[m] = color_mask
        if borders:
            import cv2
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1)

    ax.imshow(img)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
IMAGES_DIR = "/content/drive/MyDrive/nonCropPictures"

In [ ]:
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex -q
!pip install -q "paddlepaddle==3.2.2"
!pip install -q "paddleocr[all]"

from paddleocr import PaddleOCR
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False)


In [ ]:
import torch

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
start_event.record()

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)

mask_generator = SAM2AutomaticMaskGenerator(sam2)

In [ ]:
def find_mask_with_most_text(masks, rec_texts, rec_scores, rec_boxes, min_score=0.6):
    best = None

    for i, m in enumerate(masks):
        mask = m["segmentation"].astype(bool)

        char_count, texts = count_text_in_mask(
            mask, rec_texts, rec_scores, rec_boxes, min_score=min_score
        )

        if best is None or char_count > best["char_count"]:
            best = {
                "mask_idx": i,
                "char_count": char_count,
                "texts": texts,
                "mask": mask
            }

    return best

In [ ]:
def count_text_in_mask(mask_bool, rec_texts, rec_scores, rec_boxes, min_score=0.6):
    h, w = mask_bool.shape
    total_chars = 0
    kept_texts = []

    for text, score, box in zip(rec_texts, rec_scores, rec_boxes):
        if score < min_score:
            continue

        x0, y0, x1, y1 = box
        cx = int((x0 + x1) / 2)
        cy = int((y0 + y1) / 2)

        # Safety check
        if cx < 0 or cy < 0 or cx >= w or cy >= h:
            continue

        if mask_bool[cy, cx]:
            alnum = sum(ch.isalnum() for ch in text)
            total_chars += alnum
            kept_texts.append(text)

    return total_chars, kept_texts


In [ ]:
import numpy as np
import cv2

def mask_to_rotated_bbox(mask_bool):

    mask_u8 = (mask_bool.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("No contour found in mask (mask might be empty).")
    cnt = max(contours, key=cv2.contourArea)

    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)
    box = np.round(box).astype(int)

    c = box.mean(axis=0)
    angles = np.arctan2(box[:, 1] - c[1], box[:, 0] - c[0])
    box = box[np.argsort(angles)]

    start_idx = np.lexsort((box[:, 1], box[:, 0]))[0]
    box = np.roll(box, -start_idx, axis=0)

    area2 = 0
    for i in range(4):
        x1, y1 = box[i]
        x2, y2 = box[(i + 1) % 4]
        area2 += (x1 * y2 - x2 * y1)
    if area2 > 0:
        box = box[[0, 3, 2, 1]]

    return box.tolist()

In [ ]:
OUT_DIR = None


from pathlib import Path
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_rgb_uint8(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def run_paddle_ocr(rgb_uint8: np.ndarray):
    """
    PaddleOCR predict() returns:
      result = [ { 'rec_texts': [...], 'rec_scores': [...], 'rec_boxes': array(N,4), ... } ]
    """
    result = ocr.predict(rgb_uint8)
    if not result or not isinstance(result, list) or not result[0]:
        return [], [], []

    d = result[0]
    rec_texts = d.get("rec_texts", []) or []
    rec_scores = d.get("rec_scores", []) or []

    rec_boxes_arr = d.get("rec_boxes", None)
    if rec_boxes_arr is None:
        return rec_texts, rec_scores, []

    rec_boxes = rec_boxes_arr.astype(float).tolist()
    return rec_texts, rec_scores, rec_boxes

def draw_rotated_box(rgb_uint8: np.ndarray, corners, thickness: int = 3):
    vis = rgb_uint8.copy()
    pts = np.array(corners, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(vis, [pts], isClosed=True, color=(255, 0, 0), thickness=thickness)  # RGB red
    return vis

def show_image(rgb_uint8, title=None):
    plt.figure(figsize=(10, 7))
    if title:
        plt.title(title)
    plt.imshow(rgb_uint8)
    plt.axis("off")
    plt.show()

def save_mask_cutout_png(image_rgb, mask_bool, out_path, pad=10, crop=True):
    h, w = mask_bool.shape

    bgra = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGRA)

    alpha = (mask_bool.astype(np.uint8) * 255)
    bgra[..., 3] = alpha

    bgra[alpha == 0, :3] = 0

    if crop:
        ys, xs = np.where(mask_bool)
        if len(xs) == 0:
            raise ValueError("Mask is empty; nothing to export.")

        x0, x1 = xs.min(), xs.max()
        y0, y1 = ys.min(), ys.max()

        x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
        x1 = min(w - 1, x1 + pad); y1 = min(h - 1, y1 + pad)

        bgra = bgra[y0:y1+1, x0:x1+1]

    os.makedirs(os.path.dirname(str(out_path)), exist_ok=True)
    cv2.imwrite(str(out_path), bgra)

def process_and_visualize_folder(
    images_dir: str,
    min_ocr_score: float = 0.5,
    save_visuals: bool = True,
    out_dir: str = None,
    save_json: bool = True,
    save_cutouts: bool = True,
    save_masks: bool = False
):
    images_dir = Path(images_dir)
    if not images_dir.exists():
        raise ValueError(f"Folder does not exist: {images_dir}")

    files = [p for p in sorted(images_dir.iterdir())
             if p.is_file() and p.suffix.lower() in IMG_EXTS]

    if out_dir is None:
        out_dir = str(images_dir / "_boxed_outputs")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cutouts_dir = out_dir / "cutouts_png"
    masks_dir = out_dir / "masks_png"
    if save_cutouts:
        cutouts_dir.mkdir(parents=True, exist_ok=True)
    if save_masks:
        masks_dir.mkdir(parents=True, exist_ok=True)

    results = {}

    for img_path in files:
        rgb = load_rgb_uint8(img_path)

        masks = mask_generator.generate(rgb)

        rec_texts, rec_scores, rec_boxes = run_paddle_ocr(rgb)

        best = find_mask_with_most_text(
            masks=masks,
            rec_texts=rec_texts,
            rec_scores=rec_scores,
            rec_boxes=rec_boxes,
            min_score=min_ocr_score
        )

        if best is None or best.get("char_count", 0) <= 0:
            results[img_path.name] = None
            show_image(rgb, title=f"{img_path.name} — NO BOX FOUND")
            continue

        best_mask = best["mask"]

        corners = mask_to_rotated_bbox(best_mask)
        results[img_path.name] = corners

        vis = draw_rotated_box(rgb, corners, thickness=3)
        show_image(vis, title=f"{img_path.name} — chars={best['char_count']}")

        if save_visuals:
            bgr_vis = cv2.cvtColor(vis, cv2.COLOR_RGB2BGR)
            out_path = out_dir / f"{img_path.stem}_boxed.jpg"
            cv2.imwrite(str(out_path), bgr_vis)

        if save_cutouts:
            cut_path = cutouts_dir / f"{img_path.stem}_cutout.png"
            save_mask_cutout_png(rgb, best_mask, cut_path, pad=15, crop=True)

        if save_masks:
            mask_path = masks_dir / f"{img_path.stem}_mask.png"
            cv2.imwrite(str(mask_path), (best_mask.astype(np.uint8) * 255))

    if save_json:
        out_json = out_dir / "corners_results.json"
        with open(out_json, "w") as f:
            json.dump(results, f, indent=2)
        print("Saved JSON to:", str(out_json))

    print("Saved visuals to:", str(out_dir))
    if save_cutouts:
        print("Saved cutouts to:", str(cutouts_dir))
    if save_masks:
        print("Saved masks to:", str(masks_dir))

    return results, str(out_dir)

corners_by_file, out_folder = process_and_visualize_folder(
    IMAGES_DIR,
    min_ocr_score=0.5,
    save_visuals=True,
    out_dir=OUT_DIR,
    save_json=True,
    save_cutouts=True,
    save_masks=False
)

corners_by_file

In [ ]:
end_event.record()

torch.cuda.synchronize()
gpu_time_seconds = start_event.elapsed_time(end_event) / 1000.0
print(f"Total GPU Computational Time: {gpu_time_seconds:.2f} seconds")